# Bagré evapotranspiration and water stress | 01
    **Goal:** establish the study area, products, units and coverage before processing. This project measures modeled surface water consumption and exploratory stress signals, not irrigation demand or field-validated crop stress.

## 1. Connect Drive and Earth Engine
    Use a CPU runtime. Run this cell after every runtime restart. You need a registered Earth Engine Google Cloud project (not your Gmail address). Set its project ID below. The Drive folder must be in your My Drive, or a shortcut with the same name.

In [ ]:
%pip -q install earthengine-api rasterio pyproj
import ee
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Bagre_ET_Water_Stress')
assert ROOT.is_dir(), f'Open Drive and confirm this folder exists: {ROOT}'
for sub in ['data', 'outputs', 'figures', 'qgis']:
    (ROOT / sub).mkdir(exist_ok=True)
PROJECT = 'YOUR_EARTH_ENGINE_PROJECT_ID'  # CHANGE THIS
assert PROJECT != 'YOUR_EARTH_ENGINE_PROJECT_ID', 'Set your registered Cloud project ID.'
try:
    ee.Initialize(project=PROJECT)
except ee.EEException:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
print('Earth Engine connection:', ee.Number(1).getInfo())

## 2. Study design
    Reuse the approximate Bagré pilot window from the earlier water project. It includes agricultural land and other land covers; it is not an official irrigation boundary. January–April 2021 is a single dry-season demonstration. WorldCover 2021 cropland is a proxy, not a field survey. All end dates are exclusive. Do not change dates or AOI halfway through the series.

In [ ]:
import calendar
from datetime import date, timedelta
AOI_BOUNDS = [-0.59, 11.41, -0.47, 11.52]  # west, south, east, north
AOI = ee.Geometry.Rectangle(AOI_BOUNDS, proj='EPSG:4326', geodesic=False)
CENTER = AOI.centroid(1)
CRS = 'EPSG:32630'
MONTHS = [(f'2021-{m:02d}-01', f'2021-{m+1:02d}-01') for m in range(1, 5)]
S2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(AOI)
L8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterBounds(AOI)
      .filter(ee.Filter.eq('PROCESSING_LEVEL', 'L2SP')))
MOD = ee.ImageCollection('MODIS/061/MOD16A2GF').filterBounds(AOI)
WAP = ee.ImageCollection('FAO/WAPOR/3/L1_AETI_D').filterBounds(AOI)
ERA = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR').filterBounds(AOI)
WC = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map')

## 3. Check coverage
    Each row is a month. A scene count is not a count of usable pixels: cloud masking comes next. MOD16 counts represent eight-day composites and WaPOR counts represent dekads. ERA5 should have one image per calendar day. Stop if any count is zero.

In [ ]:
records = []
for start, end in MONTHS:
    row = {'month': start[:7]}
    for name, collection in [('s2', S2), ('landsat', L8), ('mod16', MOD),
                             ('wapor', WAP), ('era5', ERA)]:
        row[name] = collection.filterDate(start, end).size().getInfo()
    row['expected_era5'] = (date.fromisoformat(end) - date.fromisoformat(start)).days
    records.append(row)
audit = pd.DataFrame(records)
display(audit)
assert (audit[['s2','landsat','mod16','wapor','era5']] > 0).all().all()
assert (audit.era5 == audit.expected_era5).all()
audit.to_csv(ROOT / 'data' / 'coverage_audit.csv', index=False)
print('AOI area km²:', AOI.area(1).divide(1e6).getInfo())

### Download helper
    We download only the small pilot as GeoTIFF, with explicit CRS and scale. Earth Engine performs the processing remotely. Native thermal resolution remains about 100 m; a finer output grid would not create new information. The code checks that downloaded files have the expected number of bands and a valid coordinate system. Temporary files prevent incomplete downloads from masquerading as results.

In [ ]:
import requests
import rasterio
def download(image, path, scale, bands):
    path = Path(path)
    tmp = path.with_suffix('.partial.tif')
    url = image.select(bands).toFloat().clip(AOI).unmask(-9999, False).getDownloadURL({
        'name': path.stem, 'region': AOI, 'scale': scale,
        'crs': CRS, 'format': 'GEO_TIFF', 'filePerBand': False
    })
    with requests.get(url, stream=True, timeout=180) as response:
        response.raise_for_status()
        with tmp.open('wb') as f:
            for chunk in response.iter_content(1024 * 1024):
                f.write(chunk)
    with rasterio.open(tmp, 'r+') as src:
        assert src.count == len(bands) and src.crs.to_epsg() == 32630
        src.nodata = -9999
        for i, band in enumerate(bands, 1):
            src.set_band_description(i, band)
        print(path.name, src.shape, list(src.descriptions))
    tmp.replace(path)
    return path

## 4. Export the land-cover reference
    WorldCover code 40 is cropland. We preserve the 10 m labels for later aggregation into crop fractions. A 500 m cell dominated by cropland is not necessarily irrigated; we make no field-level claim.

In [ ]:
download(WC.rename('land_cover'), ROOT/'data'/'worldcover_2021.tif', 10, ['land_cover'])
with rasterio.open(ROOT/'data'/'worldcover_2021.tif') as src:
    arr = src.read(1, masked=True)
    plt.figure(figsize=(8, 6))
    plt.imshow(np.ma.masked_where(arr.mask, arr == 40), cmap='YlGn')
    plt.title('WorldCover 2021 cropland proxy (green)')
    plt.axis('off')
    plt.show()
    print('Cropland fraction of available pixels:', float((arr == 40).mean()))

**Checkpoint:** share the coverage table and cropland image. Are scene counts sufficient? Does the cropland pattern look plausible? If not, investigate before moving to Notebook 02.
    Sources: [WorldCover](https://developers.google.com/earth-engine/datasets/catalog/ESA_WorldCover_v200), [Sentinel-2](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED), [Landsat](https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2).